# Tải và lọc Tinix Vietnamese Job Descriptions

Nguồn: [tinixai/vietnamese-job-descriptions](https://huggingface.co/datasets/tinixai/vietnamese-job-descriptions). Dataset card công bố 606.878 bản ghi, giai đoạn 2022–2026; file nguồn là `data.parquet`. Số dòng thực tế được đọc từ file tải về. License: CC BY-NC 4.0, ghi công TiniX AI theo dataset card.

Chạy các cell từ trên xuống để tải dữ liệu và giữ những tin có **`job_title` khớp cụm từ IT, AI hoặc Data Science**. Giữ toàn bộ cột gốc, thêm `title_normalized`, `matched_groups`, `matched_phrases` để kiểm tra lý do giữ tin. Một tin có thể thuộc nhiều nhóm.

Dữ liệu gốc lưu trong `data/raw/tinixai_vietnamese_job_descriptions/`; kết quả nằm trong `data/processed/`. Cột `year` được giữ nguyên từ nguồn; không tự tạo ngày đăng đầy đủ từ năm.

## 1. Thư viện và cấu hình

Chọn Python kernel của dự án. Chạy cell cài đặt nếu thiếu thư viện; có thể thêm dấu `#` ở đầu dòng sau khi đã cài. File nguồn khoảng 293 MB theo trang Files tại thời điểm tạo notebook. Giảm `BATCH_SIZE` nếu máy ít RAM.

In [ ]:
%pip install huggingface_hub pandas pyarrow tqdm

In [ ]:
import html
import json
import re
import unicodedata
from collections import Counter
from datetime import datetime, timezone
from functools import lru_cache
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from huggingface_hub import HfApi, hf_hub_download
from IPython.display import display
from tqdm.auto import tqdm

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "notebooks").is_dir()
     and (path / "requirements.txt").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Mở notebook trong project_intro_ds hoặc notebooks/ của dự án")

REPO_ID = "tinixai/vietnamese-job-descriptions"
REVISION = "main"              # Có thể đổi thành commit SHA đầy đủ để cố định phiên bản
BATCH_SIZE = 10_000

RAW_DIR = PROJECT_ROOT / "data/raw/tinixai_vietnamese_job_descriptions"
OUTPUT_DIR = PROJECT_ROOT / "data/processed"
CSV_FILE = OUTPUT_DIR / "tinixai_it_ai_data_jobs.csv"
PARQUET_FILE = OUTPUT_DIR / "tinixai_it_ai_data_jobs.parquet"
METADATA_FILE = OUTPUT_DIR / "tinixai_it_ai_data_jobs.metadata.json"
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if BATCH_SIZE <= 0:
    raise ValueError("BATCH_SIZE phải > 0")
print("Thư mục dự án:", PROJECT_ROOT)

## 2. Các cụm từ trong title

Sửa `TITLE_PHRASES` và `TITLE_ACRONYMS` để điều chỉnh phạm vi. Cụm từ được khớp theo ranh giới từ, sau khi chuẩn hóa hoa/thường, dấu tiếng Việt và dấu phân cách. Các viết tắt ngắn được khớp trên title còn dấu để phân biệt `IT` với `ít`, `BI` với `bị`.

Nhóm Data bao gồm Data Science, Analytics, Engineering và BI. Các title chung chung như `Engineer`, `Analyst` hoặc `Data Entry` cần bổ sung cụm từ cụ thể nếu muốn giữ. Đây là bộ lọc từ khóa; xem cột `matched_phrases` và title mẫu để điều chỉnh những tin khớp chưa đúng.

In [ ]:
TITLE_PHRASES = {
    "IT": [
        "information technology", "software", "developer", "programmer",
        "frontend", "front end", "backend", "back end", "fullstack", "full stack",
        "devops", "devsecops", "platform engineer", "cloud engineer", "cloud architect",
        "cloud computing", "system engineer", "systems engineer", "system administrator",
        "systems administrator", "network engineer", "network administrator", "network architect",
        "database administrator", "database engineer", "database developer",
        "cybersecurity", "cyber security", "information security", "security engineer",
        "security analyst", "pentester", "penetration tester", "software tester",
        "manual tester", "automation tester", "qa engineer", "qa automation", "test automation",
        "helpdesk", "help desk", "it support", "it business analyst", "business systems analyst",
        "embedded", "firmware", "webmaster", "bridge system engineer", "solution architect",
        "solutions architect", "technical architect", "sap consultant", "erp consultant",
        "python", "java", "javascript", "typescript", "php", "golang", "c++", "c#",
        ".net developer", ".net engineer", "dotnet", "react", "reactjs", "nodejs", "node.js",
        "vuejs", "angular", "flutter", "kotlin", "android developer", "ios developer",
        "công nghệ thông tin", "lập trình", "phần mềm", "quản trị mạng",
        "quản trị hệ thống", "quản trị cơ sở dữ liệu", "an ninh mạng", "an toàn thông tin",
        "bảo mật thông tin", "kiểm thử phần mềm", "hỗ trợ kỹ thuật it", "kỹ sư cầu nối",
    ],
    "AI": [
        "artificial intelligence", "machine learning", "deep learning", "computer vision",
        "natural language processing", "generative ai", "large language model",
        "large language models", "mlops", "llmops", "gen ai", "prompt engineer",
        "trí tuệ nhân tạo", "học máy", "học sâu", "thị giác máy tính", "xử lý ngôn ngữ tự nhiên",
    ],
    "Data Science": [
        "data science", "data scientist", "data analyst", "data analytics", "data analysis",
        "data engineer", "data engineering", "data architect", "data architecture",
        "data warehouse", "data warehousing", "data platform", "data modeling", "data modelling",
        "data governance", "data quality", "analytics engineer", "analytics engineering",
        "business intelligence", "power bi", "powerbi", "bi developer", "bi analyst",
        "bi engineer", "bi specialist", "reporting analyst", "big data", "bigdata", "etl", "elt",
        "khoa học dữ liệu", "phân tích dữ liệu", "phân tích số liệu", "kỹ sư dữ liệu",
        "kiến trúc dữ liệu", "kho dữ liệu", "khai phá dữ liệu",
    ],
}
TITLE_ACRONYMS = {
    "IT": ["it", "cntt", "sre", "dba", "cto", "ciso", "brse"],
    "AI": ["ai", "ml", "nlp", "llm", "genai"],
    "Data Science": ["bi"],
}

In [ ]:
def normalize_title(text):
    text = unicodedata.normalize("NFKC", html.unescape(text)).casefold()
    text = "".join(char for char in unicodedata.normalize("NFD", text)
                   if unicodedata.category(char) != "Mn").replace("đ", "d")
    return re.sub(r"[^a-z0-9+#]+", " ", text).strip()


def phrase_pattern(phrases, *, normalized):
    terms = {normalize_title(term) if normalized else term.casefold().strip() for term in phrases}
    terms.discard("")
    if not terms:
        return re.compile(r"(?!)")
    choices = "|".join(re.escape(term) for term in sorted(terms, key=lambda term: (-len(term), term)))
    boundary = r"[a-z0-9+#]" if normalized else r"\w"
    return re.compile(rf"(?<!{boundary})(?:{choices})(?!{boundary})")


TITLE_RULES = {
    group: (
        phrase_pattern(phrases, normalized=True),
        phrase_pattern(TITLE_ACRONYMS.get(group, []), normalized=False),
    )
    for group, phrases in TITLE_PHRASES.items()
}


@lru_cache(maxsize=20_000)
def match_title(title):
    original = unicodedata.normalize("NFKC", html.unescape(title)).casefold()
    normalized = normalize_title(title)
    groups, matched = [], []
    for group, (phrases, acronyms) in TITLE_RULES.items():
        hits = set(phrases.findall(normalized)) | set(acronyms.findall(original))
        if hits:
            groups.append(group)
            matched.extend(f"{group}: {term}" for term in sorted(hits))
    return normalized, " | ".join(groups), "; ".join(matched)


# Chạy lại cell này và cell lọc sau khi sửa danh sách cụm từ.
print("Số cụm từ:", {group: len(terms) for group, terms in TITLE_PHRASES.items()})

## 3. Tải dataset gốc

`hf_hub_download` tải file về thư mục raw và dùng metadata cache để tái sử dụng file đã có. Cell xác định commit SHA rồi tải đúng phiên bản đó. Chạy lại cell tải khi muốn cập nhật `main`. Không cần cài thư viện `datasets`.

Tham khảo [hướng dẫn tải của Hugging Face](https://huggingface.co/docs/huggingface_hub/guides/download).

In [ ]:
dataset_info = HfApi().dataset_info(REPO_ID, revision=REVISION)
RESOLVED_REVISION = dataset_info.sha
SOURCE_FILE = Path(hf_hub_download(
    repo_id=REPO_ID, repo_type="dataset", filename="data.parquet",
    revision=RESOLVED_REVISION, local_dir=str(RAW_DIR),
))
print("Đã tải:", SOURCE_FILE)
print("Phiên bản:", RESOLVED_REVISION)
print(f"Dung lượng: {SOURCE_FILE.stat().st_size / 1024**2:,.1f} MiB")

In [ ]:
source = pq.ParquetFile(SOURCE_FILE)
SOURCE_COLUMNS = source.schema_arrow.names
EXTRA_COLUMNS = ["title_normalized", "matched_groups", "matched_phrases"]
if "job_title" not in SOURCE_COLUMNS:
    raise ValueError(f"Dataset thiếu job_title; các cột hiện có: {SOURCE_COLUMNS}")
if set(EXTRA_COLUMNS) & set(SOURCE_COLUMNS):
    raise ValueError("Dataset đã có cột trùng với các cột giải thích bộ lọc")
print(f"Số bản ghi gốc: {source.metadata.num_rows:,}")
print("Các cột:", SOURCE_COLUMNS)
sample = next(source.iter_batches(batch_size=5, use_threads=False), None)
if sample is not None:
    display(sample.to_pandas(ignore_metadata=True)[[name for name in ["id", "job_title", "company_name", "year"]
                              if name in SOURCE_COLUMNS]])

## 4. Lọc theo title và lưu kết quả

Đọc và ghi từng batch, không nạp toàn bộ JD vào RAM. Mỗi tin chỉ ghi một lần dù khớp nhiều nhóm. Chạy lại cell này để lọc lại từ nguồn sau khi sửa cụm từ; file kết quả được thay bằng lần lọc mới khi hoàn tất.

CSV dùng UTF-8 BOM để mở với Excel. Parquet giữ kiểu dữ liệu để dùng với pandas. Nếu cell lỗi hoặc bị Interrupt, các file `.tmp` chưa phải kết quả hoàn chỉnh; chạy lại cell lọc. Các file kết quả cũ được giữ đến khi lượt lọc thành công.

In [ ]:
source = pq.ParquetFile(SOURCE_FILE)
output_schema = source.schema_arrow.remove_metadata()
for name in EXTRA_COLUMNS:
    output_schema = output_schema.append(pa.field(name, pa.string()))

csv_temp = CSV_FILE.with_name(CSV_FILE.name + ".tmp")
parquet_temp = PARQUET_FILE.with_name(PARQUET_FILE.name + ".tmp")
metadata_temp = METADATA_FILE.with_name(METADATA_FILE.name + ".tmp")
source_rows = kept_rows = 0
group_counts = Counter()
year_source = Counter()
year_kept = Counter()
preview_parts = []
preview_count = 0
match_title.cache_clear()

with pq.ParquetWriter(parquet_temp, output_schema, compression="snappy") as writer, \
        csv_temp.open("w", encoding="utf-8-sig", newline="") as csv_handle, \
        tqdm(total=source.metadata.num_rows, desc="Lọc title", unit="tin") as progress:
    pd.DataFrame(columns=SOURCE_COLUMNS + EXTRA_COLUMNS).to_csv(csv_handle, index=False)
    for batch in source.iter_batches(batch_size=BATCH_SIZE, use_threads=False):
        frame = batch.to_pandas(ignore_metadata=True)
        matches = pd.DataFrame(
            [match_title(title) for title in frame["job_title"].fillna("").astype(str)],
            columns=EXTRA_COLUMNS, index=frame.index,
        )
        mask = matches["matched_groups"].ne("")
        selected = pd.concat([frame.loc[mask], matches.loc[mask]], axis=1)
        source_rows += len(frame)
        kept_rows += len(selected)
        if "year" in SOURCE_COLUMNS:
            year_source.update(frame["year"].astype("string").fillna("Thiếu năm"))
            year_kept.update(selected["year"].astype("string").fillna("Thiếu năm"))
        for groups in selected["matched_groups"]:
            group_counts.update(groups.split(" | "))
        if not selected.empty:
            writer.write_table(pa.Table.from_pandas(selected, schema=output_schema, preserve_index=False))
            selected.to_csv(csv_handle, index=False, header=False)
            if preview_count < 30:
                preview_parts.append(selected.head(30 - preview_count))
                preview_count += len(preview_parts[-1])
        progress.update(len(frame))

metadata = {
    "dataset": REPO_ID, "source_url": f"https://huggingface.co/datasets/{REPO_ID}",
    "revision": RESOLVED_REVISION, "source_file": "data.parquet", "license": "CC-BY-NC-4.0",
    "filtered_at_utc": datetime.now(timezone.utc).isoformat(), "filter_column": "job_title",
    "title_phrases": TITLE_PHRASES, "title_acronyms": TITLE_ACRONYMS,
    "source_rows": source_rows, "kept_rows": kept_rows,
    "group_counts": dict(group_counts), "year_source": dict(year_source), "year_kept": dict(year_kept),
}
metadata_temp.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
parquet_temp.replace(PARQUET_FILE)
csv_temp.replace(CSV_FILE)
metadata_temp.replace(METADATA_FILE)
print(f"Đã giữ {kept_rows:,}/{source_rows:,} tin ({kept_rows / source_rows:.2%})" if source_rows else "Dataset rỗng")
print("CSV:", CSV_FILE)
print("Parquet:", PARQUET_FILE)
print("Cụm từ và thống kê:", METADATA_FILE)

## 5. Xem kết quả và điều chỉnh bộ lọc

Số tin theo nhóm có thể cộng lớn hơn tổng số tin vì một title có thể khớp nhiều nhóm. Bảng theo năm phản ánh dữ liệu của nguồn và bộ lọc hiện tại.

Xem các title và cụm từ đã khớp bên dưới. Sửa danh sách ở phần 2 rồi chạy lại cell tạo hàm và cell lọc nếu cần. Các title thiếu từ khóa hoặc viết sai có thể bị bỏ sót.

In [ ]:
display(pd.DataFrame([
    {"Nhóm": group, "Số tin khớp": group_counts[group]} for group in TITLE_RULES
]))
if year_source:
    year_summary = pd.DataFrame([
        {"Năm trong nguồn": year, "Tin gốc": year_source[year], "Tin được giữ": year_kept[year]}
        for year in sorted(year_source)
    ])
    year_summary["Tỷ lệ giữ"] = year_summary["Tin được giữ"] / year_summary["Tin gốc"]
    display(year_summary)
if preview_parts:
    preview = pd.concat(preview_parts, ignore_index=True)
    display(preview[[name for name in ["id", "job_title", "company_name", "year",
                                      "matched_groups", "matched_phrases"]
                     if name in preview.columns]])
else:
    print("Không có title khớp. Kiểm tra lại các cụm từ và tên cột title.")

## 6. Đọc file đã lọc ở bước xử lý tiếp theo

Cell mẫu bên dưới nạp toàn bộ **kết quả đã lọc** vào RAM khi bạn cần làm EDA hoặc tiền xử lý. Bỏ dấu `#` để chạy. File raw đã tải được giữ nguyên.

In [ ]:
# jobs = pd.read_parquet(PARQUET_FILE)
# display(jobs.head())
# Hoặc: jobs = pd.read_csv(CSV_FILE)